# Lab 10: Application of Transformers for Text Classification and NER

**Lab Objective:** This lab focuses on utilizing state-of-the-art Transformer models for real-world NLP tasks. Unlike Lab 09 where we built a shallow network, here we use a pre-trained BERT model and fine-tune it for specific applications.

**Key Learning Points:**
1. Integrating pre-trained Transformer backbones into PyTorch workflows.
2. Managing Transformer-specific inputs: `input_ids`, `attention_mask`, and `token_type_ids`.
3. Fine-tuning for Sequence Classification using the `[CLS]` token representation.
4. Performing Token Classification for Named Entity Recognition (NER).
5. Handling sub-word tokenization and label alignment.

## 1. Setup: Libraries and Data

We will use the `transformers` library by Hugging Face to access BERT and the `datasets` library for the IMDb movie reviews.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, AdamW
from datasets import load_dataset
from tqdm.auto import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load IMDb dataset
raw_datasets = load_dataset("imdb")

# Use a small subset to ensure training fits in the lab session
train_set = raw_datasets["train"].shuffle(seed=42).select(range(1000))
test_set = raw_datasets["test"].shuffle(seed=42).select(range(500))

## 2. Problem 1: Fine-Tuning BERT for Sentiment Analysis

### 2.1 Tokenization
Transformers cannot process raw text. We must convert text into numerical IDs. 

**Task:** Use the `AutoTokenizer` to process the text. Ensure you use `truncation=True` and `padding='max_length'` with a `max_length` of 128.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_function(examples):
    #### YOUR CODE HERE: Tokenize the 'text' field from examples ####
    
    return None # Replace with your tokenized output
    ####

tokenized_train = train_set.map(preprocess_function, batched=True)
tokenized_test = test_set.map(preprocess_function, batched=True)

# Format for PyTorch
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

train_loader = DataLoader(tokenized_train, batch_size=16, shuffle=True)
test_loader = DataLoader(tokenized_test, batch_size=16)

### 2.2 Defining the Classifier

Instead of using a high-level wrapper, you will define a PyTorch class that uses a BERT model as its core, followed by a Dropout layer and a Linear layer for binary classification.

In [ ]:
class BertSentimentClassifier(nn.Module):
    def __init__(self, model_name):
        super(BertSentimentClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        # BERT base hidden size is 768
        self.out = nn.Linear(768, 1) 
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # BERT returns several outputs. 
        # For classification, we typically use the 'pooler_output' (the representation of [CLS])
        pooled_output = outputs.pooler_output 
        
        #### YOUR CODE HERE: Pass pooled_output through dropout and the final linear layer ####
        
        return None # Replace with your final logit
        ####

model = BertSentimentClassifier("bert-base-uncased").to(device)

### 2.3 The Training Loop

**Task:** Implement the training logic. Note that for binary classification with one output neuron, you should use `BCEWithLogitsLoss`.

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()

EPOCHS = 1 # One epoch is sufficient for this lab to see convergence

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader):
        #### YOUR CODE HERE: 
        # 1. Move inputs to device
        # 2. Zero gradients
        # 3. Forward pass
        # 4. Calculate loss (be careful with label shapes, use .unsqueeze(1) if needed)
        # 5. Backward pass and optimizer step
        
        
        ####
        pass
    
    print(f"Epoch {epoch+1} finished.")

## 3. Problem 2: Named Entity Recognition (NER) & Sub-word Analysis

As discussed in lecture, BERT uses **WordPiece** tokenization. This means a single word might be split into multiple tokens.

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", device=(0 if torch.cuda.is_available() else -1))

sample_text = "The research in the Machine Learning and Computational Genomics Lab at Santa Clara University is led by Dr. Anastasiu."
entities = ner_pipeline(sample_text)

for ent in entities:
    print(ent)

### 3.1 Task: Manual Token Inspection

The model output contains keys like `entity`, `score`, and `word`. 

**Task:** Write a script to iterate through the results of the `ner_pipeline` for the text provided below. Your script must identify and print any tokens that contain `##` (indicating they are sub-words) and explain what word they belong to.

In [ ]:
complex_text = "The geological formation in Kyrgyzstan is breathtakingly complex."
#### YOUR CODE HERE ####


####

## 4. Questions and Discussion

1. **Parallelization:** Based on the lecture slides, why can Transformers process sentences faster than LSTMs/RNNs? Reference the 'Linear interaction distance' in your answer.
2. **Special Tokens:** What was the final shape of the `pooled_output` in Problem 1? Why does this specific token represent the entire sentence?
3. **Inference Challenge:** Test the model from Problem 1 on a sarcastic review (e.g., "I loved waiting two hours for the movie to finally get boring."). Does the model predict correctly? Why might Transformers struggle with deep sarcasm despite their complexity?
4. **NER Alignment:** In Problem 2, if a word is split into three sub-tokens, but only the first sub-token is labeled as an entity, how does that affect the overall accuracy of an NER system?